In [ ]:
import sys
sys.path.append("/Users/chris/Documents/higs-vision")
import torch
import numpy as np

from src.data import load_config, load_processed
from src.models import HiggsDNN
from src.explain import compute_global_shap, compute_feature_importance, compute_local_shap
import json

config = load_config("/Users/chris/Documents/higs-vision/config.yaml")
X_train, y_train, X_val, y_val, X_test, y_test, scaler = load_processed(config)

with open("/Users/chris/Documents/higs-vision/results/optuna/best_params.json") as f:
    best = json.load(f)

width = best["hidden_width"]
model = HiggsDNN(
    input_dim=28,
    hidden_dims=[width, width // 2, width // 4, width // 8],
    activation=best["activation"],
    dropout_rate=best["dropout"],
    use_batch_norm=best["batch_norm"]
)
model.load_state_dict(torch.load("/Users/chris/Documents/higs-vision/notebooks/models/dnn/final_optuna/model.pt"))

# Test on 1000 events first
shap_values, background = compute_global_shap(model, X_train, X_test[:1000])

feature_names = [
    "lepton_pT", "lepton_eta", "lepton_phi",
    "missing_energy_mag", "missing_energy_phi",
    "jet1_pT", "jet1_eta", "jet1_phi", "jet1_b-tag",
    "jet2_pT", "jet2_eta", "jet2_phi", "jet2_b-tag",
    "jet3_pT", "jet3_eta", "jet3_phi", "jet3_b-tag",
    "jet4_pT", "jet4_eta", "jet4_phi", "jet4_b-tag",
    "m_jj", "m_jjj", "m_lv", "m_jlv", "m_bb", "m_wbb", "m_wwbb"
]

importance, ranking = compute_feature_importance(shap_values, feature_names)

print("\nTop 10 features by SHAP importance:")
for i, feat in enumerate(ranking[:10]):
    print(f"  {i+1}. {feat}: {importance[feat]:.6f}")